# RAPIDS 端到端資料分析實戰

> 結合 cuDF、cuML、CuPy，完成一個完整的資料分析專案，
> 並與 pandas + scikit-learn 的 CPU 版本進行全流程效能比較。

## GPU 加速系列 Notebooks

1. [Numba CUDA 基礎](./01-Introduction-to-cuda-python-with-numba.ipynb)
2. [GPU 加速概覽與環境建置](./02-GPU-Acceleration-Overview-and-Environment-Setup.ipynb)
3. [CuPy - GPU 版 NumPy](./03-CuPy-GPU-Accelerated-NumPy.ipynb)
4. [cuDF - GPU 版 pandas](./04-cuDF-GPU-Accelerated-DataFrames.ipynb)
5. [cuML - GPU 版 scikit-learn](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb)
6. [cuGraph - GPU 版 NetworkX](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb)
7. [Dask-CUDA 多 GPU 處理](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb)
8. **[RAPIDS 端到端實戰](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb) ← 目前位置**

---
## 專案場景：電商客戶分群分析

**目標：** 分析 500 萬筆電商交易資料，進行客戶分群（RFM 分析 + K-Means），找出高價值客戶。

**Pipeline 步驟：**
1. 資料載入
2. 資料清洗與特徵工程 (RFM)
3. 標準化
4. PCA 降維
5. K-Means 分群
6. 結果分析
7. 視覺化（傳回 CPU）

我們會同時執行 **CPU 版本** 和 **GPU 版本**，比較全流程耗時。

---
## 0. 環境準備

In [ ]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

if GPU_AVAILABLE:
    import cudf
    import cuml
    from cuml.preprocessing import StandardScaler as cuStandardScaler
    from cuml import PCA as cuPCA
    from cuml import KMeans as cuKMeans

# 紀錄各階段耗時
cpu_times = {}
gpu_times = {}

---
## 1. 產生合成電商資料

In [ ]:
N = 5_000_000  # 五百萬筆交易
N_customers = 500_000  # 50 萬客戶

np.random.seed(42)

# 模擬交易資料
data = {
    '客戶ID': np.random.randint(0, N_customers, N),
    '交易日期_天數': np.random.randint(0, 365, N).astype(np.int32),  # 最近 365 天
    '交易金額': np.random.lognormal(mean=7, sigma=1.5, size=N).astype(np.float32),
    '商品類別': np.random.choice(
        ['電子', '服飾', '食品', '家居', '美妝', '運動', '書籍', '玩具'], N
    ),
    '折扣率': np.random.uniform(0.5, 1.0, N).astype(np.float32),
    '付款方式': np.random.choice(['信用卡', '轉帳', '貨到付款', '行動支付'], N),
}

print(f'合成資料: {N:,} 筆交易, {N_customers:,} 客戶')
print(f'記憶體: ~{N * 30 / 1e6:.0f} MB')

---
## 2. CPU Pipeline (pandas + scikit-learn)

In [ ]:
# ===== 步驟 1: 載入 =====
start = time.time()
pdf = pd.DataFrame(data)
cpu_times['載入'] = time.time() - start
print(f'[CPU] 載入: {cpu_times["載入"]:.4f} 秒')
pdf.head()

In [ ]:
# ===== 步驟 2: RFM 特徵工程 =====
start = time.time()

# Recency: 最近一次交易距今天數
# Frequency: 交易次數
# Monetary: 總消費金額

rfm_cpu = pdf.groupby('客戶ID').agg(
    Recency=('交易日期_天數', 'min'),      # 最近一次交易
    Frequency=('交易金額', 'count'),         # 交易次數
    Monetary=('交易金額', 'sum'),            # 總金額
    Avg_Amount=('交易金額', 'mean'),         # 平均金額
    Avg_Discount=('折扣率', 'mean'),         # 平均折扣
).reset_index()

# 額外特徵
rfm_cpu['Amount_Std'] = pdf.groupby('客戶ID')['交易金額'].std().values
rfm_cpu['Amount_Std'] = rfm_cpu['Amount_Std'].fillna(0)

cpu_times['特徵工程'] = time.time() - start
print(f'[CPU] 特徵工程: {cpu_times["特徵工程"]:.4f} 秒')
print(f'RFM 表: {rfm_cpu.shape[0]:,} 客戶 × {rfm_cpu.shape[1]} 特徵')
rfm_cpu.head()

In [ ]:
# ===== 步驟 3: 標準化 =====
start = time.time()

feature_cols = ['Recency', 'Frequency', 'Monetary', 'Avg_Amount', 'Avg_Discount', 'Amount_Std']
X_cpu = rfm_cpu[feature_cols].values.astype(np.float32)

scaler_cpu = StandardScaler()
X_cpu_scaled = scaler_cpu.fit_transform(X_cpu)

cpu_times['標準化'] = time.time() - start
print(f'[CPU] 標準化: {cpu_times["標準化"]:.4f} 秒')

In [ ]:
# ===== 步驟 4: PCA 降維 =====
start = time.time()

pca_cpu = PCA(n_components=3)
X_cpu_pca = pca_cpu.fit_transform(X_cpu_scaled)

cpu_times['PCA'] = time.time() - start
print(f'[CPU] PCA: {cpu_times["PCA"]:.4f} 秒')
print(f'解釋變異比: {pca_cpu.explained_variance_ratio_}')
print(f'累積解釋:   {pca_cpu.explained_variance_ratio_.sum():.4f}')

In [ ]:
# ===== 步驟 5: K-Means 分群 =====
start = time.time()

kmeans_cpu = KMeans(n_clusters=5, max_iter=300, n_init=1, random_state=42)
labels_cpu = kmeans_cpu.fit_predict(X_cpu_scaled)

cpu_times['K-Means'] = time.time() - start
print(f'[CPU] K-Means: {cpu_times["K-Means"]:.4f} 秒')
print(f'各群大小: {np.bincount(labels_cpu)}')

In [ ]:
# ===== 步驟 6: 結果分析 =====
start = time.time()

rfm_cpu['群組'] = labels_cpu

cluster_summary_cpu = rfm_cpu.groupby('群組').agg({
    '客戶ID': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'Avg_Amount': 'mean',
}).rename(columns={'客戶ID': '客戶數'})

cpu_times['結果分析'] = time.time() - start

cpu_total = sum(cpu_times.values())
print(f'\n===== CPU Pipeline 完成 =====')
print(f'總耗時: {cpu_total:.4f} 秒')
print(f'\n各群特徵:')
print(cluster_summary_cpu.round(2))

---
## 3. GPU Pipeline (cuDF + cuML)

In [ ]:
if GPU_AVAILABLE:
    # ===== 步驟 1: 載入 =====
    start = time.time()
    gdf = cudf.DataFrame(data)
    gpu_times['載入'] = time.time() - start
    print(f'[GPU] 載入: {gpu_times["載入"]:.4f} 秒')

    # ===== 步驟 2: RFM 特徵工程 =====
    start = time.time()

    rfm_gpu = gdf.groupby('客戶ID').agg({
        '交易日期_天數': 'min',
        '交易金額': ['count', 'sum', 'mean', 'std'],
        '折扣率': 'mean',
    })
    rfm_gpu.columns = ['Recency', 'Frequency', 'Monetary', 'Avg_Amount', 'Amount_Std', 'Avg_Discount']
    rfm_gpu = rfm_gpu.reset_index()
    rfm_gpu['Amount_Std'] = rfm_gpu['Amount_Std'].fillna(0)

    gpu_times['特徵工程'] = time.time() - start
    print(f'[GPU] 特徵工程: {gpu_times["特徵工程"]:.4f} 秒')

    # ===== 步驟 3: 標準化 =====
    start = time.time()

    X_gpu = rfm_gpu[feature_cols]
    scaler_gpu = cuStandardScaler()
    X_gpu_scaled = scaler_gpu.fit_transform(X_gpu)

    gpu_times['標準化'] = time.time() - start
    print(f'[GPU] 標準化: {gpu_times["標準化"]:.4f} 秒')

    # ===== 步驟 4: PCA 降維 =====
    start = time.time()

    pca_gpu = cuPCA(n_components=3)
    X_gpu_pca = pca_gpu.fit_transform(X_gpu_scaled)

    gpu_times['PCA'] = time.time() - start
    print(f'[GPU] PCA: {gpu_times["PCA"]:.4f} 秒')

    # ===== 步驟 5: K-Means 分群 =====
    start = time.time()

    kmeans_gpu = cuKMeans(n_clusters=5, max_iter=300, n_init=1, random_state=42)
    labels_gpu = kmeans_gpu.fit_predict(X_gpu_scaled)

    gpu_times['K-Means'] = time.time() - start
    print(f'[GPU] K-Means: {gpu_times["K-Means"]:.4f} 秒')

    # ===== 步驟 6: 結果分析 =====
    start = time.time()

    rfm_gpu['群組'] = labels_gpu
    cluster_summary_gpu = rfm_gpu.groupby('群組').agg({
        '客戶ID': 'count',
        'Recency': 'mean',
        'Frequency': 'mean',
        'Monetary': 'mean',
        'Avg_Amount': 'mean',
    }).rename(columns={'客戶ID': '客戶數'})

    gpu_times['結果分析'] = time.time() - start

    gpu_total = sum(gpu_times.values())
    print(f'\n===== GPU Pipeline 完成 =====')
    print(f'總耗時: {gpu_total:.4f} 秒')
else:
    print('需要 GPU 環境。預期全流程加速: 10-50x')

---
## 4. 效能比較總覽

In [ ]:
print('='*65)
print(f'{"步驟":^12s} | {"CPU (秒)":^12s} | {"GPU (秒)":^12s} | {"加速倍數":^10s}')
print('='*65)

if GPU_AVAILABLE:
    for step in cpu_times:
        ct = cpu_times[step]
        gt = gpu_times.get(step, float('nan'))
        speedup = ct / gt if gt > 0 else float('nan')
        print(f'{step:^12s} | {ct:>10.4f}s | {gt:>10.4f}s | {speedup:>8.1f}x')

    print('-'*65)
    cpu_total = sum(cpu_times.values())
    gpu_total = sum(gpu_times.values())
    print(f'{"總計":^12s} | {cpu_total:>10.4f}s | {gpu_total:>10.4f}s | {cpu_total/gpu_total:>8.1f}x')
else:
    for step, ct in cpu_times.items():
        print(f'{step:^12s} | {ct:>10.4f}s | {"N/A":^12s} | {"N/A":^10s}')
    print('-'*65)
    print(f'{"總計":^12s} | {sum(cpu_times.values()):>10.4f}s | {"N/A":^12s} | {"N/A":^10s}')

---
## 5. 視覺化（傳回 CPU）

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'Noto Sans CJK TC', 'sans-serif']

    # 用 CPU 的結果視覺化（GPU 結果也可以 .to_pandas() 後使用）
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 取前 50000 筆做散佈圖（全部太密）
    sample_idx = np.random.choice(len(X_cpu_pca), min(50000, len(X_cpu_pca)), replace=False)

    # PCA 散佈圖
    scatter = axes[0].scatter(
        X_cpu_pca[sample_idx, 0], X_cpu_pca[sample_idx, 1],
        c=labels_cpu[sample_idx], cmap='viridis', s=1, alpha=0.5
    )
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].set_title('PCA + K-Means Clustering')
    plt.colorbar(scatter, ax=axes[0], label='Cluster')

    # 各群客戶數
    cluster_counts = np.bincount(labels_cpu)
    axes[1].bar(range(len(cluster_counts)), cluster_counts, color='steelblue')
    axes[1].set_xlabel('Cluster')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Customers per Cluster')

    # 效能比較
    if GPU_AVAILABLE:
        steps = list(cpu_times.keys())
        x = range(len(steps))
        width = 0.35
        axes[2].bar([i - width/2 for i in x], [cpu_times[s] for s in steps],
                    width, label='CPU', color='salmon')
        axes[2].bar([i + width/2 for i in x], [gpu_times.get(s, 0) for s in steps],
                    width, label='GPU', color='seagreen')
        axes[2].set_xticks(list(x))
        axes[2].set_xticklabels(steps, rotation=45, ha='right')
        axes[2].set_ylabel('Time (s)')
        axes[2].set_title('CPU vs GPU per Step')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'GPU data\nnot available', ha='center', va='center',
                     transform=axes[2].transAxes, fontsize=14)
        axes[2].set_title('CPU vs GPU (N/A)')

    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib 未安裝，跳過視覺化。')

---
## 6. 最佳實踐總結

### 何時使用 GPU 加速

| 條件 | 建議 |
|------|------|
| 資料 < 10 萬筆 | pandas / sklearn (CPU 足夠快) |
| 資料 10-100 萬筆 | cuDF / cuML (開始有明顯加速) |
| 資料 > 100 萬筆 | cuDF / cuML (強烈建議) |
| 資料 > GPU VRAM | Dask-CUDA (分區 / 多 GPU) |
| 超參數調校 | cuML (讓 GridSearchCV 可行) |
| 即時 ETL | cuDF (低延遲處理) |

### Pipeline 設計原則

1. **全程 GPU** — 載入 → 清洗 → 特徵 → 訓練 → 預測，全用 RAPIDS
2. **最後才轉 CPU** — 只在視覺化或輸出時 `.to_pandas()`
3. **使用 float32** — GPU float32 遠快於 float64，精度通常足夠
4. **Parquet 優先** — `cudf.read_parquet()` 比 `read_csv()` 快數倍
5. **適時釋放記憶體** — `del df` + 記憶體池清理
6. **考慮 Zero-Code-Change** — `cudf.pandas` / `cuml.accel` 零成本嘗試

### 決策流程圖

```
新的資料分析專案
      ↓
資料量 > 10 萬筆？ ─── 否 → pandas + sklearn
      │
      是
      ↓
有 NVIDIA GPU？ ─── 否 → Colab / Kaggle / Cloud GPU
      │
      是
      ↓
想改程式碼嗎？
      ├─ 不想 → cudf.pandas + cuml.accel (零改動)
      └─ 可以 → cuDF + cuML (完整控制)
              ↓
      資料 > VRAM？ ─── 是 → Dask-CUDA
              ↓
              否 → 單 GPU 即可
```

---
## 7. 系列回顧

| Notebook | 重點 | 工具 |
|----------|------|------|
| 01 | CUDA + Numba 基礎 | Numba |
| 02 | GPU 加速概覽、環境建置 | nvidia-smi, conda |
| 03 | 陣列運算加速 | CuPy |
| 04 | DataFrame 加速 | cuDF |
| 05 | 機器學習加速 | cuML |
| 06 | 圖形分析加速 | cuGraph |
| 07 | 多 GPU / 大資料 | Dask-CUDA |
| 08 | 端到端實戰 | cuDF + cuML |

### 進階學習資源

- [RAPIDS 官方文件](https://docs.rapids.ai/)
- [NVIDIA Deep Learning Institute](https://www.nvidia.com/en-us/training/) — 免費 RAPIDS 課程
- [RAPIDS GitHub](https://github.com/rapidsai) — 原始碼與範例
- [Kaggle RAPIDS 範例](https://www.kaggle.com/search?q=rapids+cudf) — 社群實戰
- [Polars GPU Engine](https://docs.pola.rs/user-guide/gpu-support/) — 新一代 DataFrame

> **恭喜完成 GPU 加速系列！** 現在你已經掌握了 RAPIDS 生態系的核心工具，
> 可以在日常資料分析工作中靈活運用 GPU 加速。